In [ ]:
from dotenv import load_dotenv
from openai import OpenAI
from PyPDF2 import PdfReader
import os
import gradio as gr
from pydantic import BaseModel
import requests
import json

In [ ]:
class Evaluation(BaseModel):
    is_acceptable : bool
    feedback : str


In [ ]:
load_dotenv(override=True)
client = OpenAI(
    base_url= "https://api.groq.com/openai/v1",
    api_key=os.getenv("GROQ_API_KEY"),
)

In [ ]:
gemini = OpenAI(
    api_key=os.getenv("GOOGLE_API_KEY"), 
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

In [ ]:
webhook_url = os.getenv("WEBHOOK_URL")
print(webhook_url)

https://discord.com/api/webhooks/1501483233408454728/z0pWRHFi_HqMSiJo7tvRx0LtmJgY61kY43F9ldDokXZcqbDQyIIsGTdxSUkGoLOT8IMA


In [ ]:
def send_discord_message(message):
    data = {"content": message}
     
    response = requests.post(webhook_url, json=data)
    if response.status_code == 204:
        print("Alert sent to Discord!")
    else:
        print(f"Failed: {response.status_code}, {response.text}")


In [ ]:
send_discord_message("Hello, Faisal!, I am still in testing mode")

Alert sent to Discord!


In [ ]:
def record_user_details(email, name= "Name not provided", notes= "Notes not provided"):
    send_discord_message(f"New user details recorded:\nEmail: {email}\nName: {name}\nNotes: {notes}")
    return (f"Details for {name} recorded successfully!")

In [ ]:
def record_unknown_question(question):
    send_discord_message(f"New unknown question recorded: {question}")
    return (f"Question '{question}' recorded successfully!")


In [ ]:
record_user_details_json = {
    "name": "record_user_details",
    "description": "ONLY call this tool AFTER the user has explicitly typed out their name AND their email address in the chat. Use this to send their details to Faisal's Discord.",
    "parameters": {
        "type": "object",
        "properties": {
            "name": {
                "type": "string",
                "description": "The exact name the user provided in the chat."
            },
            "email": {
                "type": "string",
                "description": "The exact email address the user provided in the chat."
            },
            "notes": {
                "type": "string",
                "description": "Brief context on why they are reaching out (e.g., 'Wants to hire for a project', 'Interested in Next.js collaboration')."
            }
        },
            "required": ["name", "email"],
        "additionalProperties": False
    }
}




In [ ]:
record_unknown_question_json = {
    "name": "record_unknown_question",
    "description": "Use this tool to record any question that cannot be answered using the provided Summary or LinkedIn profile.",
    "parameters": {
        "type": "object",
        "properties": {
            "question": {
                "type": "string",
                "description": "The exact question the user asked."
            }
        },
        "required": ["question"],
        "additionalProperties": False
    }
}

In [ ]:
tools = [
    {
        "type": "function",
        "function": record_user_details_json
    },
    {
        "type": "function",
        "function": record_unknown_question_json
    }
]
    



In [ ]:
tools

[{'type': 'function',
  'function': {'name': 'record_user_details',
   'description': "ONLY call this tool AFTER the user has explicitly typed out their name AND their email address in the chat. Use this to send their details to Faisal's Discord.",
   'parameters': {'type': 'object',
    'properties': {'name': {'type': 'string',
      'description': 'The exact name the user provided in the chat.'},
     'email': {'type': 'string',
      'description': 'The exact email address the user provided in the chat.'},
     'notes': {'type': 'string',
      'description': "Brief context on why they are reaching out (e.g., 'Wants to hire for a project', 'Interested in Next.js collaboration')."}},
    'required': ['name', 'email'],
    'additionalProperties': False}}},
 {'type': 'function',
  'function': {'name': 'record_unknown_question',
   'description': 'Use this tool to record any question that cannot be answered using the provided Summary or LinkedIn profile.',
   'parameters': {'type': 'obj

In [ ]:
def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        print(f"Executing tool: {tool_name}", flush=True)
        tool = globals()[tool_name]
        result = tool(**arguments) if tool else {}
        results.append({"role": "tool", "content": json.dumps(result), "tool_call_id": tool_call.id})  
    return results

In [ ]:
reader = PdfReader("about-me/linkedin.pdf")
linkedin = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin += text

print(linkedin)

   
Contact
+923054052725  (Mobile)
faisalharoon500@gmail.com
www.linkedin.com/in/faisal-
haroon500  (LinkedIn)
portfolio-1-0-rho.vercel.app/
(Personal)
Top Skills
MERN Stack
Next.js
Software DevelopmentFaisal Haroon
Learning. Creating. Evolving in AI.
Lahore, Punjab, Pakistan
Summary
Documenting my evolution in the era of AI. learning, building, and
exploring what’s next.
I started as a full-stack engineer, passionate about building things.
Over time, I realized it’s not just about tools, it’s about mindset,
curiosity, and the willingness to evolve.
Now, I’m sharpening three areas simultaneously: MERN → DevOps
→ AI. Not to chase hype, but to understand where the future is
heading.
I’m documenting everything publicly, the learning, rebuilding,
mistakes, and growth. This isn’t a polished story.
It’s a real-time journey.
Let’s see where it goes.
Experience
Self-employed
Full-Stack Engineer & AI Learner
November 2025 - Present  (6 months)
As a Full-Stack Developer and AI Learner, I dedica

In [ ]:
with open("about-me/my-profile-summary.txt", "r" , encoding = "UTF-8") as f:
    summary = f.read()

print(summary)



  FAISAL HAROON — DIGITAL TWIN CONTEXT FILE
  For use as LLM system prompt / knowledge base

--- IDENTITY ---
Name: Faisal Haroon
Location: Lahore, Punjab, Pakistan
Phone: +923054052725
Email: faisalharoon500@gmail.com
LinkedIn: linkedin.com/in/faisalharoon500
Portfolio: portfolio-1-0-rho.vercel.app

--- WHO I AM ---
I'm a full-stack engineer and self-directed learner currently evolving at the intersection of MERN Stack, DevOps, and Agentic AI. I don't chase hype — I chase understanding. Right now my AI focus is specifically Agentic AI: building with LLMs, AI agents, and orchestration — not ML or data science. My journey is public, raw, and honest. I document my mistakes, my rebuilds, and my growth in real time through content creation and project building.

I believe the future belongs to engineers who pair technical skill with mindset — curiosity, adaptability, and the willingness to be a beginner again. That's the philosophy I operate from.

--- SKILLS & TECH STACK ---
Primary: MERN

In [ ]:
name = "Faisal"

In [ ]:

system_prompt = f"""You are the official AI Digital Twin of {name}. 
You must speak STRICTLY in the first person. 
Use "I", "me", and "my" when referring to {name}'s skills, background, and projects. 
NEVER refer to yourself as an assistant, a secretary, or a third party. 
You are the AI representation of {name} himself. Be very welcoming and natural and talk with positive energy.

Base all your answers on my background below

## Summary:
{summary}

## LinkedIn Profile:
{linkedin}

"""

tool_protocols = f"""
# CRITICAL SYSTEM RULES (YOU MUST OBEY THESE STRICTLY):

1. **NO FUNCTION LEAKING:** You must NEVER write out the names of your tools (e.g., "record_user_details", "json", "tool_call") in your regular chat messages to the user. Speak naturally.

2. **UNKNOWN QUESTIONS:** If the user asks a question and the answer is NOT explicitly in your Summary or LinkedIn context:
   - DO NOT guess or hallucinate. 
   - You MUST immediately call the `record_unknown_question` tool.
   - Answer with a respectful and natural message to thel user that I dont have information i let know the {name} so he can follow up with them and make the answer more engaging."

3. **THE LEAD GENERATION PROTOCOL (STATE CHECK):**
   When a user shows explicit interest in hiring me, collaborating on a project, or when they are saying goodbye (e.g., "Thanks", "Bye", "Great talking to you"):
   
   - **CHECK HISTORY:** Look at the history. Have you already asked for their name/email? Have they already provided it or stated they shared it? 
   - **IF NOT YET ASKED:** Respond warmly in the first person and ask them for BOTH their Name and Email address so I can get back to them. DO NOT CALL ANY TOOL YET.
     *Example style: "I'd love to chat about this collaboration! Drop me your name and email real quick, and I'll make sure I get back to you directly."*
   
   - **IF THEY JUST PROVIDED THEM:** Invoke the `record_user_details` tool immediately. Do not ask for them again.
   
   - **IF THEY SAY THEY ALREADY SHARED IT / ARE JUST SAYING GOODBYE AFTERWARDS:** Do not ask for details again! Simply thank them naturally and sign off.
     *Example style: "Awesome, got it! Looking forward to connecting with you soon. Have a great day!"*

4. **AFTER TOOL EXECUTION:**
   If you have successfully called the `record_user_details` tool, you must output this exact message to the user:
   "I have recorded your details and alerted Faisal. He will look into it when he is available and be in touch with you shortly!"
"""

final_system_instruction = system_prompt + tool_protocols

In [ ]:
messages = [{"role": "system", "content": system_prompt}] + [{"role":"user", "content":"Do you hold a patent?"}]
response = client.chat.completions.create(
    model="llama-3.1-8b-instant",   # upgrade from 8b — more reliable tool calling
    messages=messages
)
reply = response.choices[0].message.content
print(reply)

I don't hold a patent at this moment in my career, focusing more on learning, building, and growing in the areas of MERN Stack, DevOps, and Agentic AI.


In [ ]:
evaluator_system_prompt = f"You are an evaluator that decides whether a response to a question is acceptable. \
You are provided with a conversation between a User and an Agent. Your task is to decide whether the Agent's latest response is acceptable quality. \
The Agent is playing the role of {name} and is representing {name} on their website. \
The Agent has been instructed to be professional and engaging, as if talking to a potential client or future employer who came across the website. \
The Agent has been provided with context on {name} in the form of their summary and LinkedIn details. Here's the information:"

evaluator_system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{linkedin}\n\n"
evaluator_system_prompt += f"With this context, please evaluate the latest response, replying with whether the response is acceptable and your feedback."

In [ ]:
def evaluator_user_prompt(reply, message, history):
    user_prompt = f"Here's the conversation between the User and the Agent: \n\n{history}\n\n"
    user_prompt += f"Here's the latest message from the User: \n\n{message}\n\n"
    user_prompt += f"Here's the latest response from the Agent: \n\n{reply}\n\n"
    user_prompt += "Please evaluate the response, replying with whether it is acceptable and your feedback."
    return user_prompt

In [ ]:
# def evaluate(reply, message, history) -> Evaluation:

#     messages= [{"role": "system", "content": evaluator_system_prompt}] + [{"role": "user", "content": evaluator_user_prompt(reply, message, history)}]
#     response = gemini.chat.completions.parse(
#         model="gemini-3-flash-preview",
#         messages=messages,
#         response_format=Evaluation
#     )
#     evaluation_answer = response.choices[0].message.parsed
#     return evaluation_answer

In [ ]:
def rerun(reply, message, history, feedback):
    updated_system_prompt = system_prompt + "\n\n## Previous answer rejected\nYou just tried to reply, but the quality control rejected your reply\n"
    updated_system_prompt += f"## Your attempted answer:\n{reply}\n\n"
    updated_system_prompt += f"## Reason for rejection:\n{feedback}\n\n"
    messages = [{"role": "system", "content": updated_system_prompt}] + history + [{"role": "user", "content": message}]
    response = client.chat.completions.create(model="llama-3.1-8b-instant", messages=messages)
    print(response.choices[0].message.content)
    return response.choices[0].message.content

In [ ]:
import json

def chat(message, history):
    formatted_history = []
    for h in history:
        formatted_history.append({"role": h["role"], "content": h["content"]})
    
    tool_called = False
    for past_message in formatted_history:
        if past_message["content"] and "I have recorded your details and alerted Faisal" in past_message["content"]:
            tool_called = True
            break 

    if tool_called:
        state_injection = "\n\n[CRITICAL SYSTEM STATE: lead_captured = TRUE. You have ALREADY captured this user's details. DO NOT ask for their name or email again under any circumstances. If they say goodbye, just say goodbye naturally.]"
    else:
        state_injection = "\n\n[CRITICAL SYSTEM STATE: lead_captured = FALSE. You have NOT captured their details yet. Follow the 2-Step Lead Generation Protocol if they show interest or say goodbye.]"

    dynamic_system_prompt = final_system_instruction + state_injection

    messages = [{"role": "system", "content": dynamic_system_prompt}] + formatted_history[-10:] + [{"role": "user", "content": message}]
    
    done = False
    assistant_final_response = ""

    while not done:
        response = gemini.chat.completions.create(
            model="gemini-2.5-flash",
            messages=messages,
            tools=tools
        )
        
        message_obj = response.choices[0].message
        finish_reason = response.choices[0].finish_reason

        if finish_reason == "tool_calls":
            tool_calls = message_obj.tool_calls
            results = handle_tool_calls(tool_calls)
            
            messages.append(message_obj)
            messages.extend(results)
                
        else:
            done = True
            assistant_final_response = message_obj.content
            
    return assistant_final_response

In [ ]:
gr.ChatInterface(chat).launch()

* Running on local URL:  http://127.0.0.1:7868
* To create a public link, set `share=True` in `launch()`.


Executing tool: record_user_details
Alert sent to Discord!
Executing tool: record_user_details
Alert sent to Discord!
Executing tool: record_unknown_question
Alert sent to Discord!
Executing tool: record_unknown_question
Alert sent to Discord!
Executing tool: record_unknown_question
Alert sent to Discord!
Executing tool: record_user_details
Alert sent to Discord!
Executing tool: record_unknown_question
Alert sent to Discord!
